In [ ]:
from google.colab import files # so you can upload files
import pandas as pd # to use data tables

In [ ]:
""" commented bc quicker upload done below - uncomment for quicker testing
# running this cell will ask user to upload the data file
# later we can change it to directly get file from repo
RAW_ABUNDANCE_DATA = files.upload()
"""

' commented bc quicker upload done below - uncomment for quicker testing\n# running this cell will ask user to upload the data file\n# later we can change it to directly get file from repo\nRAW_ABUNDANCE_DATA = files.upload()\n'

In [ ]:
from google.colab import files # upload taxa abundance file here
uploaded = files.upload()

Saving Taxa_abundance_by_sample.csv to Taxa_abundance_by_sample.csv


In [ ]:
from google.colab import files # upload sawkill mapping file here
uploaded = files.upload()

Saving Sawkill_mapping_and_env_var.csv to Sawkill_mapping_and_env_var.csv


In [ ]:
ABUNDANCE_DATA = pd.read_csv('/content/Taxa_abundance_by_sample.csv')

In [ ]:
# print(ABUNDANCE_DATA.iloc[5])
print(ABUNDANCE_DATA.shape)

(123, 3353)


In [ ]:
print(ABUNDANCE_DATA.iloc[:, 11].sum())
print(ABUNDANCE_DATA.iloc[:, 11].shape)
print(ABUNDANCE_DATA.iloc[:, 0].dtype)

33.0
(123,)
object


In [ ]:
# get only the genus that appear a significant number of times
SIGNIFICANT_SUM = 20000

significantDataList = [] # we can turn this standard list into a pd dataframe later, but it is inefficient to add columns one by one to a dataframe, so we use a list instead

for i in range(1, ABUNDANCE_DATA.shape[1]):
  column = ABUNDANCE_DATA.iloc[:, i]
  # next line checks if the sum is a number and then if that sum is great enough to be considered
  if isinstance(column.sum(), (int, float)) and column.sum() > SIGNIFICANT_SUM:
    parts = column.name.split(";g__") # column.name is the entire taxinomical name, .split separates the name and splits the phrase when ;g appears
    genus = parts[1].split(";")[0] if len(parts) > 1 else "unclassified" # checks if a list has more than 1 part to make sure after a split the split actually happens
    print(genus) # now print the genus name
    # genus = column.name.split(";g__")[1].split(";s__")[0]
    significantDataList.append(column)

Vicinamibacteraceae
Sporichthyaceae
hgcI_clade
Rhodoluna
Sediminibacterium
Pseudarcicella
Fluviicola
Fluviicola
Flavobacterium
Chloroplast
Nitrospira
Nitrospira
Hyphomicrobium
uncultured
Polynucleobacter
unclassified
Hydrogenophaga
Limnohabitans
Rhodoferax
uncultured
TRA3-20
TRA3-20


In [ ]:
print(significantDataList[0][100:])
print(len(significantDataList))

100    294.0
101    141.0
102    357.0
103    522.0
104    548.0
105    303.0
106      8.0
107      0.0
108      0.0
109     15.0
110      0.0
111     96.0
112    327.0
113    510.0
114    760.0
115    346.0
116    320.0
117    492.0
118    641.0
119      0.0
120      0.0
121      0.0
122    831.0
Name: d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__, dtype: float64
22


In [ ]:
# turn the python list of Series back into a pandas dataframe
significantData = pd.concat(significantDataList, axis=1)
print(significantData.shape)

(123, 22)


In [ ]:
# now get the answer data
ENVIRONMENT_DATA = pd.read_csv('/content/Sawkill_mapping_and_env_var.csv')

In [ ]:
print(ENVIRONMENT_DATA.shape)

(130, 26)


In [ ]:
# 1. Set the index of significantData to match the sample names from ABUNDANCE_DATA
significantData.index = ABUNDANCE_DATA['index']

# 2. Set the index of ENVIRONMENT_DATA to be its 'SampleID' column
ENVIRONMENT_DATA_indexed = ENVIRONMENT_DATA.set_index('SampleID')

# 3. Align ENVIRONMENT_DATA to match the exact order of samples present in significantData
# We use .reindex() so that we get matching rows in the exact same order.
ENVIRONMENT_DATA_aligned = ENVIRONMENT_DATA_indexed.reindex(significantData.index)

# Verify that both dataframes now have matching shapes and identical row indices
print("significantData shape:", significantData.shape)
print("Aligned ENVIRONMENT_DATA shape:", ENVIRONMENT_DATA_aligned.shape)
print("Are indices exactly identical?", significantData.index.equals(ENVIRONMENT_DATA_aligned.index))

significantData shape: (123, 22)
Aligned ENVIRONMENT_DATA shape: (123, 25)
Are indices exactly identical? True


In [ ]:
sampleRowNum = 25
print(f"Environment Data: f{ENVIRONMENT_DATA_aligned.iloc[sampleRowNum]}")
print(f"Genus Result: f{significantData.iloc[sampleRowNum]}")

Environment Data: fsort                           13.0
FieldID             B1S1_2015.06.22
Date                      6/22/2015
Day                             1.0
Site                           B1S1
GeoLocation                      B1
Type                       Sediment
Desc            Below Outflow (far)
Pool                  Below Outflow
Rain12                          0.0
Rain24                     0.125984
Rain36                     1.677162
Rain48                     1.677162
Rain72                     1.677162
AirTemp                        30.0
WaterTemp                       NaN
Turbidity                   0.00902
Conductivity                   15.0
DO_mgl                        198.0
salinity_ppt                    2.0
intI1                           NaN
Ecoli                           NaN
Coliform                        NaN
Entero                          NaN
Endotoxins                      NaN
Name: SK-13, dtype: object
Genus Result: fd__Bacteria;p__Acidobacteriota;c__Vicin

In [ ]:
print(significantData.index.equals(ENVIRONMENT_DATA_aligned.index)) # checks whether the rows align between the two charts

True


In [ ]:
significantData.to_csv("significant_genus_abundance.csv") # optional step to download the cleaned data from the taxonomy records

from google.colab import files
files.download ("significant_genus_abundance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
ENVIRONMENT_DATA_aligned.to_csv("environmental_variables.csv") # optional step to download the the correctly aligned rows of environmental variables

from google.colab import files
files.download ("environmental_variables.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
combinedData = pd.concat([significantData, ENVIRONMENT_DATA_aligned],axis=1) # defines "combined data" as to combination of the rows of ENVIRONMENT_DATA and significantData as there rows have alreayd been matched

In [ ]:
print(combinedData.columns.tolist())

['d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__Sporichthyaceae;s__metagenome', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__hgcI_clade;__', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Micrococcales;f__Microbacteriaceae;g__Rhodoluna;s__uncultured_bacterium', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Sediminibacterium;__', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Cytophagales;f__Spirosomaceae;g__Pseudarcicella;s__uncultured_bacterium', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;__', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;s__uncultured_Flexibacter', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales

In [ ]:
print(combinedData.shape) # 123 rows and 47 columns shown
print(combinedData.head()) # show first 5 rows of new combined dataset
print(combinedData.index) # display row indexes (the sample identifiers)

(123, 47)
        d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__  \
index                                                                                                                               
SK-1                                                  8.0                                                                           
SK-10                                               587.0                                                                           
SK-100                                                0.0                                                                           
SK-101                                                0.0                                                                           
SK-102                                              291.0                                                                           

        d__Bacteria;p__Actinobacteriota;c__Actinobacteria;

In [ ]:
print(combinedData.isnull().sum()) # identifies missing values in each column

d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__             0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__Sporichthyaceae;s__metagenome                   0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__hgcI_clade;__                                   0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Micrococcales;f__Microbacteriaceae;g__Rhodoluna;s__uncultured_bacterium          0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Sediminibacterium;__                             0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Cytophagales;f__Spirosomaceae;g__Pseudarcicella;s__uncultured_bacterium                 0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;__                                  0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Fl

In [ ]:
combinedData = combinedData.dropna(axis=1, thresh=len(combinedData) - 50)
combinedData = combinedData.drop(columns=["Date"])

print(combinedData.isnull().sum())

d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__            0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__Sporichthyaceae;s__metagenome                  0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__hgcI_clade;__                                  0
d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Micrococcales;f__Microbacteriaceae;g__Rhodoluna;s__uncultured_bacterium         0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Sediminibacterium;__                            0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Cytophagales;f__Spirosomaceae;g__Pseudarcicella;s__uncultured_bacterium                0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;__                                 0
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobact

In [ ]:
print(combinedData.columns.tolist())

['d__Bacteria;p__Acidobacteriota;c__Vicinamibacteria;o__Vicinamibacterales;f__Vicinamibacteraceae;g__Vicinamibacteraceae;__', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__Sporichthyaceae;s__metagenome', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Frankiales;f__Sporichthyaceae;g__hgcI_clade;__', 'd__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Micrococcales;f__Microbacteriaceae;g__Rhodoluna;s__uncultured_bacterium', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Chitinophagales;f__Chitinophagaceae;g__Sediminibacterium;__', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Cytophagales;f__Spirosomaceae;g__Pseudarcicella;s__uncultured_bacterium', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;__', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Crocinitomicaceae;g__Fluviicola;s__uncultured_Flexibacter', 'd__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales

In [ ]:
combinedData.to_csv("combined_dataset.csv") # finally we have our combined data set ready to download!!
from google.colab import files
files.download("combined_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>